In [ ]:
### If running on colab then run this cell
!pip install rank_bm25

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
import json
import pandas as pd
from pathlib import Path
import random
import re
from rank_bm25 import BM25Okapi
random.seed(42)

In [ ]:
#### Final Code for finder triplets

def normalize(s):
    return re.sub(r"\s+", " ", str(s).strip())

def tokenize(s):
    return normalize(s).lower().split()

def parse_refs(x):
    try:
        return json.loads(x.replace("'", '"'))
    except:
        return []


df = pd.read_csv("finder_augmented.csv")

df["text_x"] = df["text_x"].apply(normalize)
df["company_name"] = df["company_name"].astype(str)
df["references"] = df["references"].apply(parse_refs)

finder_queries = df.to_dict(orient="records")

finder_companies = set(df["company_name"].dropna().unique())
print(f"Found {len(finder_companies)} companies in FINdER dataset:", finder_companies)

all_chunks = json.loads(Path("chunks_index.json").read_text())

filtered_chunks = [ch for ch in all_chunks if ch["filename"] in finder_companies]

print(f"Loaded {len(all_chunks)} chunks total → "
      f"{len(filtered_chunks)} chunks after filtering by FINdER companies.")

company_to_chunks = {}
company_to_bm25 = {}

for ch in filtered_chunks:
    comp = ch["filename"]
    company_to_chunks.setdefault(comp, []).append(ch)


for comp, chunks in company_to_chunks.items():
    texts = [normalize(ch["text"]) for ch in chunks]
    tokenized = [tokenize(t) for t in texts]
    company_to_bm25[comp] = BM25Okapi(tokenized)

print(f"BM25 indices built for {len(company_to_bm25)} companies.")


# Create company-specific corpus lookups
company_to_ids = {
    comp: [ch["doc_id"] for ch in chunks]
    for comp, chunks in company_to_chunks.items()
}

company_chunk_lookup = {
    comp: {ch["doc_id"]: ch for ch in chunks}
    for comp, chunks in company_to_chunks.items()
}


def choose_positive_chunk_within_company(query_obj, company):
    refs = query_obj.get("references", [])
    refs_joined = " ".join(refs)
    ref_tokens = set(tokenize(refs_joined))

    chunks = company_to_chunks[company]

    best_chunk = None
    best_score = -1

    for ch in chunks:
        tokens = set(tokenize(ch["text"]))
        score = len(tokens & ref_tokens)
        if score > best_score:
            best_score = score
            best_chunk = ch

    return best_chunk


NUM_NEGATIVES = 3
NEG_TOP_K = 30 


def sample_negatives(query_text, company, pos_doc_id):
    bm25 = company_to_bm25[company]
    chunk_ids = company_to_ids[company]
    lookup = company_chunk_lookup[company]

    q_tokens = tokenize(query_text)
    scores = bm25.get_scores(q_tokens)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)

    neg_ids = []
    for idx in ranked[:NEG_TOP_K]:
        cid = chunk_ids[idx]
        if cid != pos_doc_id:
            neg_ids.append(cid)
        if len(neg_ids) >= NUM_NEGATIVES:
            break


    if len(neg_ids) < NUM_NEGATIVES:
        other = []
        for other_comp, other_ids in company_to_ids.items():
            if other_comp == company:
                continue
            other.extend(other_ids)

        random.shuffle(other)
        needed = NUM_NEGATIVES - len(neg_ids)
        neg_ids.extend(other[:needed])

    negatives = []
    for nid in neg_ids:
        # identify company from nid
        comp = nid.split("::")[0]
        ch = company_chunk_lookup[comp][nid]
        negatives.append({ "chunk_id": nid, "text": ch["text"], "company_name": comp})

    return negatives


triplets = []

for q in finder_queries:
    qid = q["_id"]
    company = q["company_name"]
    query_text = q["text_x"]

    if company not in company_to_chunks:
      print(f"[SKIP] No chunks found for company {company}, skipping query {qid}")
      continue

    pos = choose_positive_chunk_within_company(q, company)

    if pos is None:
        print(f"[WARN] No positive found for query {qid} ({company})")
        continue

    pos_doc_id = pos["doc_id"]

    negatives = sample_negatives(query_text, company, pos_doc_id)

    triplets.append({
        "query_id": qid,
        "query": query_text,
        "company_name": company,
        "positive": {
            "chunk_id": pos_doc_id,
            "text": pos["text"],
            "company_name": company
        },
        "negatives": negatives
    })


out_path = "finder_triplets_optimized.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for t in triplets:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")

print(f"\nSaved {len(triplets)} optimized triplets → {out_path}")


Found 293 companies in FINdER dataset: {'FTV', 'BAC', 'MLM', 'CRM', 'VTRS', 'GEN', 'BG', 'VZ', 'EXC', 'SWK', 'KEYS', 'LULU', 'TSLA', 'HPQ', 'RSG', 'CTAS', 'NXPI', 'PPG', 'CNP', 'NWSA', 'AMAT', 'VRSN', 'AME', 'BEN', 'BBY', 'MS', 'PSA', 'PM', 'NVDA', 'ATO', 'CSX', 'CHRW', 'BXP', 'META', 'CVX', 'WDC', 'AMZN', 'CMG', 'SRE', 'CTVA', 'SBUX', 'PYPL', 'CARR', 'TSCO', 'PNC', 'APTV', 'IEX', 'PHM', 'WAB', 'MMM', 'AIG', 'SMCI', 'CTSH', 'STLD', 'LVS', 'LH', 'COST', 'PKG', 'NI', 'MNST', 'APH', 'NTAP', 'PTC', 'WBD', 'HII', 'LIN', 'MPWR', 'QCOM', 'PANW', 'WTW', 'HOLX', 'DD', 'DOW', 'HON', 'ETR', 'MCK', 'OKE', 'BLDR', 'FANG', 'NVR', 'KMI', 'LRCX', 'TECH', 'DLTR', 'AAPL', 'BX', 'PFG', 'NSC', 'GILD', 'FTNT', 'KDP', 'PEP', 'FOXA', 'POOL', 'INCY', 'RL', 'XYL', 'OTIS', 'HAS', 'SPGI', 'DXCM', 'KLAC', 'SJM', 'DE', 'MSCI', 'TXN', 'ARE', 'AMCR', 'PNW', 'PRU', 'ALGN', 'COP', 'XOM', 'AON', 'WYNN', 'TAP', 'SYY', 'ADI', 'CPB', 'UAL', 'PWR', 'ADM', 'VICI', 'AMGN', 'TROW', 'MSI', 'HD', 'AOS', 'HCA', 'DPZ,', 'CSCO', '